In [1]:
import sys
import os
from datetime import datetime

project_root = os.path.dirname(
    os.path.dirname(os.getcwd())
)

sys.path.append(project_root)

print(project_root)

# ============================================
# BLOCK 1 — IMPORTS
# ============================================


from pathlib import Path
from datetime import datetime, timedelta

import sqlite3
import uuid

from layer_2.database.mock_db import init_mock_db

from layer_2.repositories.sqlite_order_repository import SQLiteOrderRepository
from layer_2.repositories.customer_repository import SQLiteCustomerRepository
from layer_2.repositories.sqlite_refund_repository import SQLiteRefundRepository

from layer_2.services.mock_payment_service import MockPaymentService

from layer_2.agents.refund_agent.refund_agent_node import RefundAgentNode

from layer_2.schemas.refund_models import RefundRequest

print("Imports successful.")

print(f"✅ Imports fixed. Project root recognized.")

c:\Users\Shiwan\OneDrive\Desktop\multi_agent_customer_support_system
Imports successful.
✅ Imports fixed. Project root recognized.


In [2]:
# ============================================
# BLOCK 2 — RESET DATABASE
# ============================================

init_mock_db(force_reset=True)

print("Database reset complete.")

🗑️ Old database cleared.
✅ Production-Grade Relational DB initialized at: c:\Users\Shiwan\OneDrive\Desktop\multi_agent_customer_support_system\layer_2\database\ecommerce.db
Database reset complete.


In [3]:
# ============================================
# BLOCK 3 — RESOLVE DATABASE PATH
# ============================================

BASE_DIR = Path.cwd().parent.parent

DB_PATH = BASE_DIR / "layer_2" / "database" / "ecommerce.db"

print("Resolved DB Path:")
print(DB_PATH)

print("\nDatabase Exists?")
print(DB_PATH.exists())

Resolved DB Path:
c:\Users\Shiwan\OneDrive\Desktop\multi_agent_customer_support_system\layer_2\database\ecommerce.db

Database Exists?
True


In [4]:
# ============================================
# BLOCK 4 — CREATE REPOSITORIES
# ============================================

order_repo = SQLiteOrderRepository(str(DB_PATH))

customer_repo = SQLiteCustomerRepository(str(DB_PATH))

refund_repo = SQLiteRefundRepository(str(DB_PATH))

payment_service = MockPaymentService()

print("Repositories initialized.")

Repositories initialized.


In [5]:
# ============================================
# BLOCK 5 — CREATE REFUND AGENT
# ============================================

refund_agent = RefundAgentNode(
    order_repo=order_repo,
    customer_repo=customer_repo,
    refund_repo=refund_repo,
    payment_service=payment_service
)

print("Refund Agent initialized.")

Refund Agent initialized.


In [6]:
# ============================================
# BLOCK 6 — INSERT SAFE SUCCESS TEST DATA
# ============================================

conn = sqlite3.connect(str(DB_PATH))

cursor = conn.cursor()

# Trusted old customer
old_customer_date = (
    datetime.now() - timedelta(days=365)
).isoformat()

cursor.execute(
    """
    INSERT OR REPLACE INTO customers
    (
        customer_id,
        total_spent,
        refund_count,
        account_tier,
        created_at
    )
    VALUES (?, ?, ?, ?, ?)
    """,
    (
        "USER-TRUSTED",
        2500.00,
        0,
        "GOLD",
        old_customer_date
    )
)

# Safe refundable order
purchase_date = (
    datetime.now() - timedelta(days=5)
).isoformat()

cursor.execute(
    """
    INSERT OR REPLACE INTO orders
    (
        order_id,
        customer_id,
        amount,
        purchase_date,
        status,
        is_refundable
    )
    VALUES (?, ?, ?, ?, ?, ?)
    """,
    (
        "ORD-SAFE-001",
        "USER-TRUSTED",
        120.00,
        purchase_date,
        "delivered",
        True
    )
)

conn.commit()
conn.close()

print("Safe success test data inserted.")

Safe success test data inserted.


In [7]:
# ============================================
# TEST 1 — SUCCESSFUL REFUND FLOW
# ============================================

request = RefundRequest(
    ticket_id="TICKET-SUCCESS-001",
    customer_id="USER-TRUSTED",
    order_id="ORD-SAFE-001",
    reason_for_refund="Damaged item"
)

decision = refund_agent.process_refund_request(
    request=request,
    idempotency_key=str(uuid.uuid4())
)

print("\nFINAL DECISION:")
print(decision)


FINAL DECISION:
status=<RefundStatus.COMPLETED: 'completed'> code='REFUND_COMPLETED' reason='Refund processed successfully.' refund_amount=120.0 requires_human_review=False metadata={'transaction_id': 'TXN-3C3F0304'}


In [8]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(DB_PATH)

df = pd.read_sql_query("SELECT * FROM processed_refunds", conn)
conn.close()

# Display the audit trail
df

,idempotency_key,order_id,refund_status,decision_reason,refund_amount,requires_human_review,metadata,created_at
0,bc3f432c-67bc-4eac-a0af-a5ee48804046,ORD-SAFE-001,completed,Refund processed successfully.,120.0,0,"{""transaction_id"": ""TXN-3C3F0304""}",2026-05-10 09:37:30


In [9]:
# ============================================
# TEST 2 — IDEMPOTENCY REPLAY
# ============================================

same_key = str(uuid.uuid4())

request = RefundRequest(
    ticket_id="TICKET-IDEMPOTENT-001",
    customer_id="USER-TRUSTED",
    order_id="ORD-SAFE-001",
    reason_for_refund="Retry simulation"
)

# First Request
decision_1 = refund_agent.process_refund_request(
    request=request,
    idempotency_key=same_key
)

print("\nFIRST RESPONSE:")
print(decision_1)

# Replay Same Request
decision_2 = refund_agent.process_refund_request(
    request=request,
    idempotency_key=same_key
)

print("\nSECOND RESPONSE (IDEMPOTENT REPLAY):")
print(decision_2)


FIRST RESPONSE:
status=<RefundStatus.COMPLETED: 'completed'> code='REFUND_COMPLETED' reason='Refund processed successfully.' refund_amount=120.0 requires_human_review=False metadata={'transaction_id': 'TXN-38912F3F'}

SECOND RESPONSE (IDEMPOTENT REPLAY):
status=<RefundStatus.COMPLETED: 'completed'> code='IDEMPOTENT_REPLAY' reason='Refund processed successfully.' refund_amount=120.0 requires_human_review=False metadata={'transaction_id': 'TXN-38912F3F'}
